# Adding semantic tags to an sqlite table

This notebook adds semantic tags (currently from ekilex) to lemmas that appear in spatial cases in the Estonian Reference corpus. The semantic types are added to a sqlite database table *spatial_obl* into a new column called *ekilex_tag*.

In [6]:
#imports
import sqlite3
import os
import re

In [7]:
# database file path
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# connecting with database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

### Add new column for the semantic tags

In [8]:
cursor.execute("ALTER TABLE spatial_obl ADD COLUMN ekilex_tag TEXT")

### Save word and semantic type to dict

In [9]:
word_semtype = dict()  # key word, value tag

directory_str = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_location_by_context\\base_data\\ekilexist\\wordlists"
directory = os.fsencode("C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_location_by_context\\base_data\\ekilexist\\wordlists")
    
for file in os.listdir(directory):
    file_name = os.fsdecode(file)
    filepath = directory_str + '\\' + file_name
    semtype = re.findall(r"^(.+?)\.[^.]+$", file_name)
    with open(filepath, "r", encoding="utf-8") as f:
        words = f.read().splitlines()
        for word in words:
            word_semtype[word] = semtype[0]  

### Get all lemmas of obliques in spatial cases in the database table
Excludes pronouns

In [10]:
cursor.execute("SELECT lemma FROM spatial_obl")
lemmas = cursor.fetchall()
lemmas_uniq = list(set(lemmas))

### Put lemma and its semantic tag together

In [11]:
#prepare data for bulk update
update_data = [(word_semtype[word], word) for lemma in lemmas_uniq if (word := lemma[0]) in word_semtype]

### Put the lemmas and tags into a temporary table

In [12]:
# Step 1: Create a temporary table
cursor.execute("CREATE TEMP TABLE IF NOT EXISTS temp_updates (lemma TEXT PRIMARY KEY, ekilex_tag TEXT)")

# Step 2: Insert all values into the temp table
cursor.executemany("INSERT INTO temp_updates (ekilex_tag, lemma) VALUES (?, ?)", update_data)

### Add temporary table info to spatial_obl table

In [13]:
# Step 3: Perform a fast join-based update
cursor.execute("""
    UPDATE spatial_obl
    SET ekilex_tag = (SELECT ekilex_tag FROM temp_updates WHERE temp_updates.lemma = spatial_obl.lemma)
    WHERE pos != 'P' AND EXISTS (SELECT 1 FROM temp_updates WHERE temp_updates.lemma = spatial_obl.lemma)
""")

### Commit changes and close the database connection

In [14]:
conn.commit()
conn.close()